# Day 1: Bond Pricing — `price_from_ytm` and `ytm_from_price`

Tests for `src/termstructure/bonds/pricing.py` using real zero yields pulled from the GSW database.

The database stores zero yields (`sveny01`–`sveny30`), not individual bond prices.
So the approach here is: treat the database's zero yield at a given maturity as the YTM,
construct a bond with that coupon and maturity, and verify the pricing math holds.

In [ ]:
import sys
sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import numpy as np

from termstructure.io import load_bonds_for_date
from termstructure.bonds.pricing import price_from_ytm, ytm_from_price

## 1. Pull a real yield curve from the database

We use 2020-01-15 as our reference date — a normal trading day with a positively-sloped curve.

In [ ]:
df = load_bonds_for_date('2020-01-15')

# extract zero yields in decimal form
maturities = [1, 2, 3, 5, 7, 10, 20, 30]
zero_yields = {m: df[f'sveny{m:02d}'].iloc[0] / 100 for m in maturities}

print('Zero yields on 2020-01-15:')
for m, y in zero_yields.items():
    print(f'  {m:2d}Y: {y*100:.4f}%')

## 2. Par bond test

When `coupon == ytm`, a bond must price at exactly par ($100).
This is the fundamental identity of bond math.
We verify it for every maturity on the curve.

In [ ]:
print('Par bond test (coupon = ytm => price = 100):')
for m, ytm in zero_yields.items():
    price = price_from_ytm(coupon=ytm, maturity_years=m, ytm=ytm)
    err = abs(price - 100)
    status = 'OK' if err < 1e-6 else 'FAIL'
    print(f'  {m:2d}Y  ytm={ytm*100:.4f}%  price={price:.6f}  err={err:.2e}  {status}')

## 3. Round-trip test

`ytm_from_price(price_from_ytm(ytm)) == ytm` to machine precision.
This validates that the two functions are exact inverses of each other.

In [ ]:
coupon = 0.04  # fixed 4% coupon — will be off-market for most maturities

print(f'Round-trip test (coupon = {coupon*100:.0f}%, priced at real zero yields):')
for m, ytm in zero_yields.items():
    price = price_from_ytm(coupon=coupon, maturity_years=m, ytm=ytm)
    ytm_recovered = ytm_from_price(price=price, coupon=coupon, maturity_years=m)
    err = abs(ytm_recovered - ytm)
    status = 'OK' if err < 1e-8 else 'FAIL'
    print(f'  {m:2d}Y  price={price:8.4f}  ytm_in={ytm*100:.4f}%  ytm_out={ytm_recovered*100:.4f}%  err={err:.2e}  {status}')

## 4. Premium and discount bonds

Using the real 10Y zero yield as YTM:
- A bond with a **higher coupon** than YTM trades at a **premium** (> 100)
- A bond with a **lower coupon** than YTM trades at a **discount** (< 100)
- A bond with coupon **equal** to YTM trades at **par** (= 100)

In [ ]:
ytm_10y = zero_yields[10]
print(f'10Y zero yield: {ytm_10y*100:.4f}%')
print()

coupons = [0.01, 0.02, ytm_10y, 0.04, 0.06, 0.08]
print(f'{"Coupon":>8}  {"Price":>10}  {"Premium/Discount":>18}')
print('-' * 42)
for c in coupons:
    p = price_from_ytm(coupon=c, maturity_years=10, ytm=ytm_10y)
    label = 'par' if abs(p - 100) < 0.001 else ('premium' if p > 100 else 'discount')
    print(f'{c*100:7.2f}%  {p:10.4f}  {label:>18}')

## 5. Price vs yield relationship

Price and yield move inversely. This plot shows the full price-yield curve
for a 5% coupon 10Y bond, with the real 10Y market yield marked.

In [ ]:
ytm_range = np.linspace(0.001, 0.12, 300)
prices = [price_from_ytm(coupon=0.05, maturity_years=10, ytm=y) for y in ytm_range]

market_ytm = zero_yields[10]
market_price = price_from_ytm(coupon=0.05, maturity_years=10, ytm=market_ytm)

plt.figure(figsize=(9, 4))
plt.plot(ytm_range * 100, prices, color='steelblue')
plt.axvline(market_ytm * 100, color='tomato', linestyle='--', label=f'10Y market yield ({market_ytm*100:.2f}%)')
plt.scatter([market_ytm * 100], [market_price], color='tomato', zorder=5)
plt.axhline(100, color='gray', linestyle=':', linewidth=0.8, label='Par ($100)')
plt.xlabel('YTM (%)')
plt.ylabel('Price ($)')
plt.title('Price–Yield Curve: 5% coupon, 10Y bond')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'At market yield {market_ytm*100:.4f}%, a 5% coupon 10Y bond prices at ${market_price:.4f}')

## 6. Pricing across the whole curve

For each maturity on the 2020-01-15 curve, price a bond whose coupon equals the 10Y yield (0.1806%).
Bonds with maturity < 10Y trade at a discount (short rates were lower than 10Y in Jan 2020).
Bonds with maturity > 10Y trade at a premium (long rates were higher — more coupon income).

Wait — actually on Jan 15 2020 the curve was fairly flat from 1Y–5Y and then rose.
The pricing direction depends on how each maturity's zero yield compares to the fixed coupon.

In [ ]:
fixed_coupon = zero_yields[10]  # fix coupon at 10Y zero yield

prices_across_curve = []
for m, ytm in zero_yields.items():
    p = price_from_ytm(coupon=fixed_coupon, maturity_years=m, ytm=ytm)
    prices_across_curve.append((m, ytm * 100, p))

print(f'Fixed coupon = {fixed_coupon*100:.4f}% (10Y zero yield)')
print()
print(f'{"Maturity":>10}  {"Zero Yield":>12}  {"Price":>10}  {"vs Par":>10}')
print('-' * 50)
for m, ytm_pct, price in prices_across_curve:
    vs_par = price - 100
    print(f'{m:>8}Y  {ytm_pct:>10.4f}%  {price:>10.4f}  {vs_par:>+10.4f}')

## 7. Repeat on an inverted curve (2023)

In mid-2023 the curve was inverted — short rates above long rates.
A bond with a coupon equal to the 10Y yield will now trade at a *premium* at short maturities
(because short-end discount rates are higher than the coupon)
— wait, let's just run it and see what the data says.

In [ ]:
df_inv = load_bonds_for_date('2023-07-03')
zero_yields_inv = {m: df_inv[f'sveny{m:02d}'].iloc[0] / 100 for m in maturities}

fixed_coupon_inv = zero_yields_inv[10]

print('Inverted curve — 2023-07-03')
print(f'Fixed coupon = {fixed_coupon_inv*100:.4f}% (10Y zero yield)')
print()
print(f'{"Maturity":>10}  {"Zero Yield":>12}  {"Price":>10}  {"vs Par":>10}')
print('-' * 50)
for m, ytm in zero_yields_inv.items():
    p = price_from_ytm(coupon=fixed_coupon_inv, maturity_years=m, ytm=ytm)
    print(f'{m:>8}Y  {ytm*100:>10.4f}%  {p:>10.4f}  {p-100:>+10.4f}')

## 8. FRED CMT yields — par and round-trip tests

The FRED data (`data/processed/cmt_yields.parquet`) stores constant-maturity Treasury yields
at 6 tenors: 3M, 1Y, 2Y, 5Y, 10Y, 30Y. These are on-the-run benchmark yields, not zero rates.

One caveat: the 3M tenor is a T-bill — a zero-coupon discount instrument, not a coupon bond.
Our `price_from_ytm` assumes semi-annual coupons, so we skip 3M here and only test true coupon maturities.

In [ ]:
import pandas as pd

fred = pd.read_parquet('../data/processed/cmt_yields.parquet')
fred['date'] = pd.to_datetime(fred['date'])

date = '2020-01-15'
fred_day = fred[fred['date'] == date].set_index('maturity_years')['yield_pct'] / 100
fred_day = fred_day.drop(index=0.25)  # skip 3M T-bill

print(f'FRED CMT yields on {date}:')
for mat, ytm in fred_day.items():
    print(f'  {mat:.0f}Y: {ytm*100:.4f}%')

print()
print('Par bond test:')
for mat, ytm in fred_day.items():
    price = price_from_ytm(coupon=ytm, maturity_years=mat, ytm=ytm)
    err = abs(price - 100)
    status = 'OK' if err < 1e-6 else 'FAIL'
    print(f'  {mat:.0f}Y  ytm={ytm*100:.4f}%  price={price:.6f}  err={err:.2e}  {status}')

print()
print('Round-trip test (coupon = 4%):')
for mat, ytm in fred_day.items():
    price = price_from_ytm(coupon=0.04, maturity_years=mat, ytm=ytm)
    ytm_rec = ytm_from_price(price=price, coupon=0.04, maturity_years=mat)
    err = abs(ytm_rec - ytm)
    status = 'OK' if err < 1e-8 else 'FAIL'
    print(f'  {mat:.0f}Y  price={price:8.4f}  ytm_in={ytm*100:.4f}%  ytm_out={ytm_rec*100:.4f}%  err={err:.2e}  {status}')

## 9. FRED CMT vs GSW zero yields — same date comparison

The two sources measure different things:
- **FRED CMT** yields are par yields — the coupon rate that prices a bond exactly at $100 for each maturity
- **GSW zero** yields are zero-coupon yields — the rate for a single cash flow at that maturity

On a normal upward-sloping curve, zero yields sit *above* par yields at long maturities
(because par yields blend in near-term cash flows discounted at lower short rates).
This plot verifies that relationship holds in our data.

In [ ]:
gsw_mats = [1, 2, 5, 10, 30]
gsw_yields = [zero_yields[m] * 100 for m in gsw_mats]

fred_mats = list(fred_day.index)
fred_yields = [v * 100 for v in fred_day.values]

plt.figure(figsize=(9, 4))
plt.plot(gsw_mats, gsw_yields, marker='o', label='GSW zero yields (Fed)', color='steelblue')
plt.plot(fred_mats, fred_yields, marker='s', label='FRED CMT par yields', color='tomato', linestyle='--')
plt.xlabel('Maturity (years)')
plt.ylabel('Yield (%)')
plt.title(f'GSW Zero vs FRED CMT Yields — {date}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Spread (GSW zero - FRED CMT) at shared maturities:')
shared = [m for m in gsw_mats if m in fred_mats]
for m in shared:
    diff = zero_yields[m] * 100 - fred_day[m] * 100
    print(f'  {m:2d}Y  {diff:+.4f} bps')